# 10 - CTD controlled evaluation for evidence-grounded reasoning

This notebook is a controlled follow-up to 09. It uses the same positive/negative prompt template for both supported-path and no-path cases, so the label cannot be inferred from wording such as `if no...`.

The evaluation includes:
- clean positive paths
- positive paths with irrelevant distractors
- hard no-path cases where the true disease name appears in a distractor edge
- lexical distractors using gene-like but incorrect symbols
- counterfactual evidence where the supplied disease is deliberately changed

Two pilot training conditions are compared:
1. Vanilla SFT: positive clean paths only
2. Robust+Abstain SFT: positive clean + distractors + no-path examples

This notebook is self-contained and uses a small pilot setup for Colab/L4.

In [ ]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"


In [ ]:
import os, re, random
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
CHEM_GENE='/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE='/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    print('Upload CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz')
    files.upload()
assert os.path.exists(CHEM_GENE) and os.path.exists(GENE_DISEASE)
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
chem_cols=['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols=['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']
chem=pd.read_csv(CHEM_GENE,sep='\t',comment='#',header=None,names=chem_cols,dtype=str,low_memory=False)
gd=pd.read_csv(GENE_DISEASE,sep='\t',comment='#',header=None,names=gd_cols,dtype=str,low_memory=False)
chem=chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].dropna(subset=['ChemicalName','ChemicalID','GeneSymbol','GeneID']).copy()
gd=gd.dropna(subset=['GeneID','DiseaseName','DiseaseID']).drop_duplicates(['GeneID','DiseaseID']).copy()
chem['GeneID']=chem['GeneID'].str.replace(r'\.0$','',regex=True); gd['GeneID']=gd['GeneID'].str.replace(r'\.0$','',regex=True)
pairs=chem.merge(gd[['GeneID','DiseaseName','DiseaseID']],on='GeneID',how='inner')
pairs=pairs[['ChemicalName','ChemicalID','GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna().drop_duplicates(['ChemicalID','GeneID','DiseaseID']).reset_index(drop=True)
print('2-hop paths:',len(pairs))


In [ ]:
# Chemical-disjoint pilot split.
rng=random.Random(42)
chems=pairs['ChemicalID'].drop_duplicates().tolist(); rng.shuffle(chems)
eval_chems=set(chems[:max(1,int(.1*len(chems)))])
train_pool=pairs[~pairs['ChemicalID'].isin(eval_chems)].sample(frac=1,random_state=42).head(1800).reset_index(drop=True)
eval_pool=pairs[pairs['ChemicalID'].isin(eval_chems)].sample(frac=1,random_state=43).head(100).reset_index(drop=True)
edge_pool=list({(str(g),str(d)) for g,d in pairs[['GeneSymbol','DiseaseName']].itertuples(index=False,name=None)})
gene_pool=list({str(g) for g,_ in edge_pool})
print('Train:',len(train_pool),'Eval:',len(eval_pool),'Edges:',len(edge_pool))


In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
MODEL_NAME='Qwen/Qwen2.5-0.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def render(prompt,answer=None):
    msgs=[{'role':'user','content':prompt}]
    if answer is not None: msgs.append({'role':'assistant','content':answer})
    return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=answer is None)


In [ ]:
# One controlled template is used for every example. Only the graph evidence changes.
def controlled_prompt(chemical, gene, edges):
    edge_text='\n'.join(f'- {g} -> {d}' for g,d in edges)
    return (f'Chemical: {chemical}\n'
            f'Queried gene: {gene}\n'
            f'Gene-disease evidence:\n{edge_text}\n'
            'Question: What disease, if any, is supported for the chemical through the queried gene? '
            'Return exactly one of: `Disease: <name>` or `No supported path.`')

YES_PREFIX='Disease:'
NO_ANSWER='No supported path.'

def positive_item(row, distractors=0, rng=None):
    rng=rng or random.Random(0)
    edges=[(row.GeneSymbol,row.DiseaseName)]
    if distractors:
        cand=[x for x in edge_pool if x[0]!=row.GeneSymbol and x[1]!=row.DiseaseName]
        for x in rng.sample(cand,min(distractors,len(cand))): edges.append(x)
    rng.shuffle(edges)
    return controlled_prompt(row.ChemicalName,row.GeneSymbol,edges), f'Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'

def hard_no_path_same_disease(row, distractors=3, rng=None):
    rng=rng or random.Random(0)
    # Include the true disease name through a different gene, but never the queried gene.
    cand=[x for x in edge_pool if x[0]!=row.GeneSymbol and x[1]==row.DiseaseName]
    others=[x for x in edge_pool if x[0]!=row.GeneSymbol and x[1]!=row.DiseaseName]
    edges=[]
    if cand: edges.append(rng.choice(cand))
    if len(edges)<distractors: edges.extend(rng.sample(others,min(distractors-len(edges),len(others))))
    rng.shuffle(edges)
    return controlled_prompt(row.ChemicalName,row.GeneSymbol,edges), NO_ANSWER

def lexical_no_path(row, distractors=4, rng=None):
    rng=rng or random.Random(0)
    # Use gene symbols with lexical overlap where possible, but not the queried gene.
    prefix=row.GeneSymbol[:2].upper()
    lex=[g for g in gene_pool if g!=row.GeneSymbol and g.startswith(prefix)]
    lex=lex[:distractors] if len(lex)>=distractors else rng.sample([g for g in gene_pool if g!=row.GeneSymbol],min(distractors,len(gene_pool)-1))
    edges=[(g,row.DiseaseName if i==0 else rng.choice([d for _,d in edge_pool])) for i,g in enumerate(lex)]
    rng.shuffle(edges)
    return controlled_prompt(row.ChemicalName,row.GeneSymbol,edges), NO_ANSWER

def counterfactual_item(row, rng=None):
    rng=rng or random.Random(0)
    alt=[d for d in pairs['DiseaseName'].drop_duplicates().tolist() if d!=row.DiseaseName]
    cf=rng.choice(alt)
    return controlled_prompt(row.ChemicalName,row.GeneSymbol,[(row.GeneSymbol,cf)]), f'Disease: {cf}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {cf}.', cf


In [ ]:
# Build balanced training data with the SAME controlled template.
def make_train(condition,seed):
    rr=random.Random(seed); rows=[]
    source=train_pool.sample(n=min(1800,len(train_pool)),random_state=seed).reset_index(drop=True)
    for row in source.itertuples(index=False):
        u=rr.random()
        if condition=='vanilla':
            p,a=positive_item(row,0,rr)
        else:
            if u<.4: p,a=positive_item(row,0,rr)
            elif u<.7: p,a=positive_item(row,3,rr)
            else: p,a=hard_no_path_same_disease(row,3,rr)
        rows.append({'text':render(p,a)})
    return Dataset.from_list(rows)

vanilla_ds=make_train('vanilla',1)
robust_ds=make_train('robust',2)
print('Datasets:',len(vanilla_ds),len(robust_ds))


In [ ]:
# Build controlled evaluation sets.
eval_sets={'clean':[],'distractor_5':[],'hard_no_path':[],'lexical_no_path':[],'counterfactual':[]}
rr=random.Random(123)
for row in eval_pool.itertuples(index=False):
    p,_=positive_item(row,0,rr); eval_sets['clean'].append({'prompt':p,'answer':row.DiseaseName,'type':'positive'})
    p,_=positive_item(row,5,rr); eval_sets['distractor_5'].append({'prompt':p,'answer':row.DiseaseName,'type':'positive'})
    p,a=hard_no_path_same_disease(row,5,rr); eval_sets['hard_no_path'].append({'prompt':p,'answer':None,'type':'no_path'})
    p,a=lexical_no_path(row,4,rr); eval_sets['lexical_no_path'].append({'prompt':p,'answer':None,'type':'no_path'})
    p,a,cf=counterfactual_item(row,rr); eval_sets['counterfactual'].append({'prompt':p,'answer':cf,'type':'positive'})
for k,v in eval_sets.items(): print(k,len(v))


In [ ]:
from peft import LoraConfig
from trl import SFTConfig,SFTTrainer
lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,target_modules=['q_proj','k_proj','v_proj','o_proj'],bias='none',task_type='CAUSAL_LM')

def train(ds,outdir):
    m=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda(); m.config.use_cache=False
    args=SFTConfig(output_dir=outdir,per_device_train_batch_size=4,gradient_accumulation_steps=2,max_steps=80,learning_rate=2e-4,logging_steps=20,save_strategy='no',report_to='none',packing=False,gradient_checkpointing=False,fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    t=SFTTrainer(model=m,args=args,train_dataset=ds,processing_class=tokenizer,peft_config=lora)
    t.train(); t.save_model(outdir); tokenizer.save_pretrained(outdir)
    del t,m; torch.cuda.empty_cache()

train(vanilla_ds,'./outputs/10-vanilla')
train(robust_ds,'./outputs/10-robust')


In [ ]:
from peft import PeftModel

def generate_batched(m,prompts,batch_size=16,max_new_tokens=48):
    outs=[]; m.eval()
    for s in range(0,len(prompts),batch_size):
        enc=tokenizer([render(p) for p in prompts[s:s+batch_size]],return_tensors='pt',padding=True,truncation=True,max_length=320)
        enc={k:v.to(m.device) for k,v in enc.items()}
        with torch.inference_mode(): out=m.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        n=enc['input_ids'].shape[1]; outs.extend(tokenizer.batch_decode(out[:,n:],skip_special_tokens=True))
    return outs

def normalize(x): return re.sub(r'[^a-z0-9]+',' ',str(x).lower()).strip()

def score(items,preds):
    vals=[]
    for it,p in zip(items,preds):
        q=normalize(p)
        if it['type']=='no_path': vals.append('no supported path' in q)
        else: vals.append(normalize(it['answer']) in q)
    return sum(vals)/len(vals) if vals else float('nan')

def evaluate(path,label):
    base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda(); m=PeftModel.from_pretrained(base,path); m.eval(); out={}
    for name,items in eval_sets.items():
        preds=generate_batched(m,[x['prompt'] for x in items]); out[name]=score(items,preds); print(f'{label:10s} {name:20s} {out[name]:.3f}')
    del m,base; torch.cuda.empty_cache(); return out

vanilla=evaluate('./outputs/10-vanilla','Vanilla')
robust=evaluate('./outputs/10-robust','Robust')


In [ ]:
print('\n10 CONTROLLED EVALUATION')
print('='*86)
print(f"{'Condition':<20}{'Vanilla':>14}{'Robust':>14}{'Delta':>12}")
print('-'*86)
for k in eval_sets:
    print(f"{k:<20}{vanilla[k]:>14.3f}{robust[k]:>14.3f}{robust[k]-vanilla[k]:>12.3f}")


## Interpretation

The key control is that positive and no-path cases share the same surface prompt template. The no-path label is not revealed by wording such as `if no...`; the distinction is only in the supplied graph evidence.

`hard_no_path` keeps the true disease name in the evidence through a different gene, so disease-name matching alone should fail. `lexical_no_path` introduces gene-symbol distractors with lexical overlap. `counterfactual` changes the disease attached to the queried gene and tests whether the model follows the supplied evidence rather than a prior association.

For a paper-quality result, repeat this notebook across multiple random seeds and larger evaluation sets, and add gene-disjoint and disease-disjoint splits.